In [28]:
from difflib import SequenceMatcher
import numpy as np
from ranking import utils, config
from pathlib import Path
import os

BASE_DIR = os.getcwd()

config.db_path = str(BASE_DIR + "/Data/")
config.table = "text"
config.text_col = "text"
config.normalized_col = "text"
config.id_col = "id"
config.hadith_txt = str(BASE_DIR + "/Data/diff.txt")
config.dbs = ["diff_test.db", "diff_test2.db"]
db_txt = utils.get_txt_from_db(current_db="diff_test2.db", config=config)
cand_txt, cand_meta = utils.get_cand_txt()


def diff_to_matrix(ref: str, cand: str):
    seq_res: list = []
    vec1_ids: list = []
    vec1_lens: list = []
    vec2_ids: list = []
    vec2_lens: list = []

    for list_list in SequenceMatcher(None, ref, cand).get_grouped_opcodes():
        for list_e in list_list:
            list_e = list(list_e)
            if list_e[0] == "equal":
                list_e[0] = 1
            elif list_e[0] in ["replace", "delete", "insert"]:
                list_e[0] = -1
            elif list_e != []:
                raise ValueError(f"Unexpected tag: {list_e[0]}")
            try:
                _1_ids, _1_lens = utils.char_idx_to_token_idx(
                    ref, [e for e in range(list_e[1], list_e[2] + 1)]
                )
                _2_ids, _2_lens = utils.char_idx_to_token_idx(
                    cand, [e for e in range(list_e[3], list_e[4] + 1)]
                )
                vec1_ids += _1_ids
                vec1_lens += _1_lens
                vec2_ids += _2_ids
                vec2_lens += _2_lens
            except IndexError:
                pass

            seq_res.append(list_e)

    if seq_res == []:
        return None, None, None, None, None, None
    max_len_1 = max(max(len(d) for d in p) for p in vec1_ids)
    max_len_2 = max(max(len(d) for d in p) for p in vec2_ids)
    pos_matrix_1 = np.array(
        [[d + [0] * (max_len_1 - len(d)) for d in p] for p in vec1_ids],
        dtype=np.int16,
    )
    pos_matrix_2 = np.array(
        [[d + [0] * (max_len_2 - len(d)) for d in p] for p in vec2_ids],
        dtype=np.int16,
    )
    return pos_matrix_1, pos_matrix_2, vec1_ids, vec1_lens, vec2_ids, vec2_lens


def main():
    total_time = 0

    for i, (key, val) in enumerate(cand_meta.items()):
        pos_matrix_1, pos_matrix_2, vec1_ids, vec1_lens, vec2_ids, vec2_lens = (
            diff_to_matrix(val["normalized"], db_txt[i][0])
        )
        try:
            if vec1_ids is None:
                print(f"ID:{i+1} 100% Übereinstimmung")
                continue
        except ValueError:
            pass
        pos_matrix_1 = np.vectorize(lambda i: config.POS_WEIGHTS.get(val["pos"][i]))(
            np.array([v for v in vec1_ids])
        )

        pos_matrix_2 = np.vectorize(lambda i: config.POS_WEIGHTS.get(val["pos"][i]))(
            np.array([v for v in vec1_ids])
        )

        pos_m__len_m_1 = [np.array(vec1_lens) * pos_matrix_1[vec1_ids]]
        pos_m__len_m_2 = [np.array(vec2_lens) * pos_matrix_2[vec2_ids]]
        print(pos_m__len_m_1)
        # devided_v1 = [(vec1 / max(vec1_dicts.keys()) + 1) * pos_matrix[:, 0]]
        # dot_pos_v1 = np.where(
        #     pos_matrix[:, 0] > 0,
        #     pos_matrix[:, 0]
        #     * (
        #         np.vectorize(lambda i: list(vec1_dicts[i].keys())[0])(
        #             np.arange(len(vec1))
        #         )
        #         / len(val["normalized"])
        #     ),
        #     0,
        # )
        # dot_pos_v2 = np.where(
        #     pos_matrix[:, 1] > 0,
        #     pos_matrix[:, 1]
        #     * (
        #         np.vectorize(lambda i: vec2_dicts[vec2[i]])(np.arange(len(vec2)))
        #         / len(val["normalized"])
        #     ),
        #     0,
        # )
        # dot_vecs = dot_pos_v1 + dot_pos_v2
        # positive = dot_vecs[dot_vecs > 0].sum()
        # negative = dot_vecs[dot_vecs < 0].sum()
        # percentage = positive / (positive + (negative * (-1)))
        # print(seq_matrix)
        # print(vec1, vec2)
        # print(dot_vecs)
        # print(positive)
        # print(negative)

        # print(f"ID:{i+1} {percentage*100:.2f}", "% Übereinstimmung")

    # total_time += (
    #     timeit.timeit(lambda: diff_to_matrix(row, db_txt[i]), number=1000) / 1000
    # )


# print(
#     f"{'PERFORMANCE':^20} \n",
#     f"{total_time:.4f} s",
# )


if __name__ == "__main__":
    main()


ID:1 100% Übereinstimmung
ID:2 100% Übereinstimmung
ID:3 100% Übereinstimmung
ID:4 100% Übereinstimmung


ValueError: not enough values to unpack (expected 2, got 1)